# Hyperparameter Optimization for IHDP Agent

This notebook demonstrates hyperparameter optimization for the IHDPAgent (Incremental Heuristic Dynamic Programming) using Optuna.
The objective function trains the agent on the LinearLongitudinalF16-v0 environment and returns the final reward.

## Imports and Objective Definition

Import the required libraries and define the Optuna objective function that creates the F-16 environment,
configures the IHDP actor/critic/incremental settings, runs a simulation, and returns the reward.

In [2]:
import optuna
import numpy as np
import gymnasium as gym
from tensoraerospace.agent.ihdp.model import IHDPAgent
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standard import unit_step

dt = 0.01
tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)
reference_signals = np.reshape(unit_step(degree=5, tp=tp, time_step=10*dt, output_rad=True), [1, -1])


def objective(trial):

    env = gym.make('LinearLongitudinalF16-v0',
               number_time_steps=number_time_steps,
               initial_state=[[0], [0], [0]],
               reference_signal=reference_signals,
               use_reward=False,
               state_space=["theta", "alpha", "q"],
               output_space=["theta", "alpha", "q"],
               tracking_states=["alpha"])
    env.reset()

    actor_settings = {
        "start_training": trial.suggest_int("start_training", 1, 20, log=True),
        "layers": (trial.suggest_int("layers", 1, 100, log=True), 1),
        "activations": ('tanh', 'tanh'),
        "learning_rate": trial.suggest_int("learning_rate", 1, 20, log=True),
        "learning_rate_exponent_limit": 10,
        "type_PE": "combined",
        "amplitude_3211": 15,
        "pulse_length_3211": 5 / dt,
        "maximum_input": 25,
        "maximum_q_rate": 20,
        "WB_limits": 30,
        "NN_initial": 120,
        "cascade_actor": False,
        "learning_rate_cascaded": 1.2
    }
    incremental_settings = {
        "number_time_steps": number_time_steps,
        "dt": dt,
        "input_magnitude_limits": 25,
        "input_rate_limits": 60,
    }
    critic_settings = {
        "Q_weights": [trial.suggest_float('Q_weights', 0, 100)],
        "start_training": -1,
        "gamma": trial.suggest_float('gamma', 0, 0.99),
        "learning_rate": trial.suggest_int("learning_rate_critic", 1, 20, log=True),
        "learning_rate_exponent_limit": 10,
        "layers": (trial.suggest_int("layers_critic", 1, 100, log=True), 1),
        "activations": ("tanh", "linear"),
        "WB_limits": 30,
        "NN_initial": 120,
        "indices_tracking_states": env.unwrapped.indices_tracking_states
    }

    model = IHDPAgent(actor_settings, critic_settings, incremental_settings,
                      env.unwrapped.tracking_states, env.unwrapped.state_space,
                      env.unwrapped.control_space, number_time_steps,
                      env.unwrapped.indices_tracking_states)
    xt = np.array([[0], [0], [0]])
    for step in range(number_time_steps - 3):
        ut = model.predict(xt, reference_signals, step)
        xt, reward, terminated, truncated, info = env.step(np.array(ut))
    return reward

## Create Optuna Study

Initialize the Optuna study with minimization direction.

In [3]:
study = optuna.create_study(direction="minimize")

[I 2026-04-25 23:49:08,234] A new study created in memory with name: no-name-5b9e2820-8dec-4a7f-9e1e-52ada691e0c0


## Run Optimization

Execute the optimization with 20 trials.

In [4]:
study.optimize(objective, n_trials=20)

[I 2026-04-25 23:49:11,599] Trial 0 finished with value: 1.0 and parameters: {'start_training': 7, 'layers': 2, 'learning_rate': 5, 'Q_weights': 57.207845675641025, 'gamma': 0.31855747743125395, 'learning_rate_critic': 11, 'layers_critic': 2}. Best is trial 0 with value: 1.0.
[I 2026-04-25 23:49:12,975] Trial 1 finished with value: 1.0 and parameters: {'start_training': 2, 'layers': 72, 'learning_rate': 2, 'Q_weights': 91.29235290183705, 'gamma': 0.8725928661809997, 'learning_rate_critic': 1, 'layers_critic': 19}. Best is trial 0 with value: 1.0.
[I 2026-04-25 23:49:14,348] Trial 2 finished with value: 1.0 and parameters: {'start_training': 1, 'layers': 8, 'learning_rate': 3, 'Q_weights': 37.87568426385285, 'gamma': 0.7067569229936439, 'learning_rate_critic': 6, 'layers_critic': 20}. Best is trial 0 with value: 1.0.
[I 2026-04-25 23:49:15,710] Trial 3 finished with value: 1.0 and parameters: {'start_training': 5, 'layers': 81, 'learning_rate': 5, 'Q_weights': 30.863294215763894, 'gamma

## Best Trial Results

Retrieve the best trial parameters found during optimization.

In [5]:
study.best_trial

FrozenTrial(number=0, state=1, values=[1.0], datetime_start=datetime.datetime(2026, 4, 25, 23, 49, 9, 601457), datetime_complete=datetime.datetime(2026, 4, 25, 23, 49, 11, 599188), params={'start_training': 7, 'layers': 2, 'learning_rate': 5, 'Q_weights': 57.207845675641025, 'gamma': 0.31855747743125395, 'learning_rate_critic': 11, 'layers_critic': 2}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'start_training': IntDistribution(high=20, log=True, low=1, step=1), 'layers': IntDistribution(high=100, log=True, low=1, step=1), 'learning_rate': IntDistribution(high=20, log=True, low=1, step=1), 'Q_weights': FloatDistribution(high=100.0, log=False, low=0.0, step=None), 'gamma': FloatDistribution(high=0.99, log=False, low=0.0, step=None), 'learning_rate_critic': IntDistribution(high=20, log=True, low=1, step=1), 'layers_critic': IntDistribution(high=100, log=True, low=1, step=1)}, trial_id=0, value=None)